# 01 - Baseline: text → embedding pipeline + 5 prompt → image test runs

**Project:** text-to-image generation with CLIP + Stable Diffusion v1.5 on a curated COCO subset.

This notebook covers task 4 (baseline text→embedding pipeline) and produces the "5 sample prompt→image runs" deliverable.

| Step | What happens |
|---|---|
| 0 | Setup (Colab or local) |
| 1 | Load the cleaned COCO subset and inspect it |
| 2 | Text → embedding pipeline (CLIP ViT-L/14 text encoder) |
| 3 | Embed all captions and run a retrieval sanity check on the data |
| 4 | Generate the 5 test images with zero-shot SD v1.5 |
| 5 | Score and save results to `results/baseline_v0/` |

**Before running:** `python scripts/prepare_data.py --n 2000` (creates `data/processed/`). On Colab choose *Runtime → Change runtime type → T4 GPU*.

In [ ]:
# --- Colab only: uncomment, set your repo URL, run once ---
# !git clone https://github.com/<your-username>/text2image-coco-diffusion.git
# %cd text2image-coco-diffusion
# %pip install -q -r requirements.txt
# !python scripts/prepare_data.py --n 2000

In [ ]:
import json, random, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

# Works whether the notebook is opened from the repo root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from t2i.data import load_metadata
from t2i.embed import encode_corpus, encode_text, get_device, load_text_encoder
from t2i.evaluate import clip_scores, embed_images, embed_texts, load_clip, recall_at_k
from t2i.generate import SD_MODEL_ID, generate, load_pipeline, make_grid, save_run

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = get_device()

import diffusers, transformers
print(f"device={DEVICE} | torch {torch.__version__} | diffusers {diffusers.__version__} | transformers {transformers.__version__}")
if DEVICE == "cpu":
    print("WARNING: no GPU found - Step 4 will be very slow. Use Colab T4 or reduce steps.")

## 1. Load and inspect the cleaned subset

In [ ]:
DATA = ROOT / "data" / "processed"
meta = load_metadata(DATA / "metadata.jsonl")
df = pd.DataFrame(meta)
df["n_words"] = df["caption"].str.split().str.len()

print(json.dumps(json.loads((DATA / "stats.json").read_text()), indent=2))

fig, axes = plt.subplots(2, 4, figsize=(14, 7.5))
for ax, row in zip(axes.ravel(), df.sample(8, random_state=SEED).itertuples()):
    ax.imshow(Image.open(DATA / row.file)); ax.axis("off")
    ax.set_title(row.caption[:48] + ("…" if len(row.caption) > 48 else ""), fontsize=8)
plt.tight_layout(); plt.show()

df["n_words"].hist(bins=range(4, 30), figsize=(6, 2.5)); plt.xlabel("words per primary caption"); plt.show()

## 2. Text → embedding pipeline

SD v1.x conditions its UNet on the **last hidden states** of the CLIP ViT-L/14 text encoder: a tensor of shape `[77, 768]` per prompt. The **pooled** vector (hidden state at the EOS token) is a single 768-d sentence embedding, which we use for retrieval checks and later analysis.

In [ ]:
tokenizer, text_encoder = load_text_encoder(device=DEVICE)

TEST_PROMPTS = json.loads((ROOT / "prompts" / "test_prompts.json").read_text())
prompts = [p["prompt"] for p in TEST_PROMPTS]

# Tokenisation: what the model actually sees
ids = tokenizer(prompts[0]).input_ids
print(len(ids), "tokens ->", tokenizer.convert_ids_to_tokens(ids))

enc = encode_text(prompts, tokenizer, text_encoder)
print("input_ids   :", tuple(enc["input_ids"].shape))
print("token_embeds:", tuple(enc["token_embeds"].shape), "<- UNet cross-attention input")
print("pooled      :", tuple(enc["pooled"].shape), "<- sentence-level vector")
print("truncated   :", enc["truncated"])

In [ ]:
# Do semantically similar prompts land close together? (5 test prompts + 2 paraphrases)
probe = prompts + ["a double decker bus on a wet street in the rain", "a puppy leaping for a frisbee outdoors"]
emb = torch.nn.functional.normalize(encode_text(probe, tokenizer, text_encoder)["pooled"], dim=-1)
sim = (emb @ emb.T).numpy()

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(sim, cmap="viridis"); plt.colorbar(im)
labels = [f"P{i+1}" for i in range(5)] + ["P1'", "P3'"]
ax.set_xticks(range(len(labels)), labels); ax.set_yticks(range(len(labels)), labels)
ax.set_title("Cosine similarity of pooled CLIP text embeddings"); plt.show()
print(f"paraphrase P1~P1': {sim[0, 5]:.3f} | unrelated P1~P2: {sim[0, 1]:.3f}")

## 3. Embed all captions + retrieval sanity check

Two goals: (a) cache caption embeddings for later experiments, and (b) verify the data cleaning didn't break image–caption alignment. If CLIP can retrieve the right validation image from its caption well above chance (1/N), the pairs are sound.

In [ ]:
ART = ROOT / "artifacts"; ART.mkdir(exist_ok=True)
captions = df["caption"].tolist()
caption_emb = encode_corpus(captions, tokenizer, text_encoder)
torch.save({"image_id": df["image_id"].tolist(), "pooled": caption_emb}, ART / "caption_embeddings.pt")
print("saved", tuple(caption_emb.shape), "->", ART / "caption_embeddings.pt")

In [ ]:
clip_model, clip_proc = load_clip(device=DEVICE)
val = df[df["split"] == "val"].reset_index(drop=True)
val_images = [Image.open(DATA / f).convert("RGB") for f in val["file"]]

img_emb = embed_images(val_images, clip_model, clip_proc)
txt_emb = embed_texts(val["caption"].tolist(), clip_model, clip_proc)
rec = recall_at_k(txt_emb, img_emb)
print(f"N_val={len(val)} | chance R@1 = {1/len(val):.4f}")
print({k: round(v, 3) for k, v in rec.items()})

## 4. Five prompt → image test runs (zero-shot SD v1.5)

Fixed seeds, 30 DPM-Solver++ steps, CFG 7.5, 512×512, so the later **LoRA fine-tuned** model can be compared on *identical* prompts and seeds.

In [ ]:
del text_encoder; torch.cuda.empty_cache() if DEVICE == "cuda" else None  # free VRAM before loading SD

pipe = load_pipeline(device=DEVICE)
seeds = [p["seed"] for p in TEST_PROMPTS]
images = generate(pipe, prompts, seeds, steps=30, guidance=7.5)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, im, p in zip(axes.ravel(), images, prompts):
    ax.imshow(im); ax.axis("off"); ax.set_title(p, fontsize=9, wrap=True)
axes.ravel()[-1].axis("off")
plt.tight_layout(); plt.show()

## 5. Score and save

CLIPScore = 100 × cosine(image, prompt) under CLIP ViT-L/14 (higher = better prompt adherence). Typical SD v1.5 values for simple COCO-style prompts are in the low-to-mid 30s; treat the number as a **relative** baseline, not an absolute quality grade.

In [ ]:
scores = clip_scores(images, prompts, clip_model, clip_proc)
results = pd.DataFrame({"prompt": prompts, "seed": seeds, "clip_score": np.round(scores, 2)})
display(results)
print("mean CLIPScore:", round(float(np.mean(scores)), 2))

config = dict(model=SD_MODEL_ID, scheduler="DPMSolverMultistep", steps=30, guidance=7.5,
              resolution=512, mean_clip_score=round(float(np.mean(scores)), 3))
save_run(ROOT / "results" / "baseline_v0", prompts, seeds, images, scores, config)
print("saved to", ROOT / "results" / "baseline_v0")

## Next steps
1. **Inspect failures** - which prompts did the baseline get wrong (counts, spatial relations, text)? Note them in the report.
2. **LoRA fine-tune** the UNet attention layers on the `train` split (rank 8-16) and re-run the *same* 5 prompts/seeds into `results/lora_v1/`.
3. **Evaluate** both models on the held-out `test` captions: CLIPScore, FID/KID vs. real test images, plus a small human preference check.